# 🔥 지역난방 열수요 예측: 시즌별 CatBoost 모델

## 📋 모델링 전략
- **시즌 분할**: Heating Season vs Non-Heating Season (2개 모델)
- **CatBoost**: 범주형 변수 자동 처리 및 Ordered Boosting
- **시계열 분해**: STL을 통한 트렌드/계절성 특성 추가
- **하이퍼파라미터 최적화**: Optuna TPE 사용
- **총 모델 수**: 2개 (난방시즌, 비난방시즌)

In [16]:
# Google Colab 환경 확인 및 패키지 설치
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔥 Google Colab 환경에서 실행 중...")
    !pip install catboost optuna statsmodels
    from google.colab import files, drive
    print("✅ 패키지 설치 완료!")
else:
    print("💻 로컬 환경에서 실행 중...")

💻 로컬 환경에서 실행 중...


In [17]:
# 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
import pickle
import json

# 머신러닝
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit

# CatBoost
import catboost as cb
from catboost import CatBoostRegressor

# Optuna
import optuna
from optuna.samplers import TPESampler

# 시계열 분해
from statsmodels.tsa.seasonal import STL
import holidays

plt.rcParams['figure.figsize'] = (12, 6)
print("📚 라이브러리 로드 완료! (CatBoost 포함)")
print(f"🔧 CatBoost 버전: {cb.__version__}")
print(f"🔧 Optuna 버전: {optuna.__version__}")
print(f"🔧 STL 분해 사용")

📚 라이브러리 로드 완료! (CatBoost 포함)
🔧 CatBoost 버전: 1.2.8
🔧 Optuna 버전: 4.3.0
🔧 STL 분해 사용


## 1️⃣ 데이터 로드 및 기본 전처리

In [18]:
# 데이터 파일 로드
if IN_COLAB:
    print("📁 파일 업로드 방법 선택:")
    print("1. 직접 업로드")
    print("2. Google Drive")

    method = input("선택 (1 또는 2): ")

    if method == "1":
        uploaded = files.upload()
        files_list = list(uploaded.keys())
        train_path = [f for f in files_list if 'train' in f.lower()][0]
        test_path = [f for f in files_list if 'test' in f.lower()][0]
    else:
        drive.mount('/content/drive')
        train_path = "/content/drive/MyDrive/train_data_2122_processed.csv"
        test_path = "/content/drive/MyDrive/test_data_23_processed.csv"
else:
    train_path = "train_data_2122_processed.csv"
    test_path = "test_data_23_processed.csv"

print(f"✅ 파일 경로 설정 완료")
print(f"   훈련: {train_path}")
print(f"   테스트: {test_path}")

✅ 파일 경로 설정 완료
   훈련: train_data_2122_processed.csv
   테스트: test_data_23_processed.csv


In [19]:
# 데이터 로드
print("📊 데이터 로드 중...")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 컬럼명 정리
def clean_columns(df):
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    df.columns = [col.replace('train_heat.', '') for col in df.columns]
    return df

train_df = clean_columns(train_df)
test_df = clean_columns(test_df)

# 날짜 변환
train_df['tm'] = pd.to_datetime(train_df['tm'])
test_df['tm'] = pd.to_datetime(test_df['tm'])

print(f"📈 훈련 데이터: {train_df.shape}")
print(f"📉 테스트 데이터: {test_df.shape}")
print(f"📅 훈련 기간: {train_df['tm'].min()} ~ {train_df['tm'].max()}")
print(f"📅 테스트 기간: {test_df['tm'].min()} ~ {test_df['tm'].max()}")
print(f"🏢 브랜치 수: {train_df['branch_id'].nunique()}개")
print(f"🏢 브랜치 목록: {sorted(train_df['branch_id'].unique())}")

📊 데이터 로드 중...
📈 훈련 데이터: (332861, 11)
📉 테스트 데이터: (166440, 11)
📅 훈련 기간: 2021-01-01 01:00:00 ~ 2022-12-31 23:00:00
📅 테스트 기간: 2023-01-01 00:00:00 ~ 2023-12-31 23:00:00
🏢 브랜치 수: 19개
🏢 브랜치 목록: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S']


## 2️⃣ 고급 특성 엔지니어링

### 📊 범주형 변수 정의 및 생성

In [20]:
def create_categorical_features(df):
    """범주형 변수 생성 및 정의"""
    df_processed = df.copy()
    
    print("🔧 범주형 변수 생성 중...")
    
    # wd 변수 제거 (요청사항)
    if 'wd' in df_processed.columns:
        df_processed = df_processed.drop(columns=['wd'])
        print("   ❌ wd(풍향) 변수 제거")
    
    # 기본 시간 변수
    df_processed['year'] = df_processed['tm'].dt.year
    df_processed['month'] = df_processed['tm'].dt.month
    df_processed['day'] = df_processed['tm'].dt.day
    df_processed['hour'] = df_processed['tm'].dt.hour
    df_processed['weekday'] = df_processed['tm'].dt.weekday
    df_processed['dayofyear'] = df_processed['tm'].dt.dayofyear
    
    # 🏢 브랜치 ID (범주형)
    df_processed['branch_id'] = df_processed['branch_id'].astype(str)
    
    # 📅 월 (범주형)
    df_processed['month_cat'] = df_processed['month'].astype(str)
    
    # 📅 요일 (범주형)
    weekday_names = ['월', '화', '수', '목', '금', '토', '일']
    df_processed['weekday_name'] = df_processed['weekday'].map(lambda x: weekday_names[x])
    
    # 🕐 시간대 구분 (범주형)
    def get_time_period(hour):
        if 6 <= hour < 12:
            return '오전'
        elif 12 <= hour < 18:
            return '오후'
        elif 18 <= hour < 22:
            return '저녁'
        else:
            return '새벽'
    
    df_processed['time_period'] = df_processed['hour'].map(get_time_period)
    
    # ❄️ 계절 (범주형)
    def get_season(month):
        if month in [12, 1, 2]:
            return '겨울'
        elif month in [3, 4, 5]:
            return '봄'
        elif month in [6, 7, 8]:
            return '여름'
        else:
            return '가을'
    
    df_processed['season'] = df_processed['month'].map(get_season)
    
    # 🏠 난방시즌 (수치형 - 모델 분할용으로만 사용, 범주형 변수에서는 제외)
    df_processed['heating_season'] = df_processed['month'].isin([10,11,12,1,2,3,4]).astype(int)
    
    # 🌧️ 기상청 기준 시간강수량 (현재 코드 교체)
    df_processed['rain_status'] = '없음'
    df_processed.loc[df_processed['rn_hr1'] >= 0.1, 'rain_status'] = '약함'      # 0.1~2.9mm
    df_processed.loc[df_processed['rn_hr1'] >= 3.0, 'rain_status'] = '보통'      # 3.0~14.9mm  
    df_processed.loc[df_processed['rn_hr1'] >= 15.0, 'rain_status'] = '강함'     # 15.0~29.9mm
    df_processed.loc[df_processed['rn_hr1'] >= 30.0, 'rain_status'] = '매우강함' # 30.0mm+

    # 🌧️ 기상청 기준 일강수량 (현재 코드 교체)
    df_processed['daily_rain_category'] = '없음'
    df_processed.loc[df_processed['rn_day'] >= 0.1, 'daily_rain_category'] = '약함'      # 0.1~4.9mm
    df_processed.loc[df_processed['rn_day'] >= 5.0, 'daily_rain_category'] = '보통'      # 5.0~19.9mm
    df_processed.loc[df_processed['rn_day'] >= 20.0, 'daily_rain_category'] = '강함'     # 20.0~79.9mm
    df_processed.loc[df_processed['rn_day'] >= 80.0, 'daily_rain_category'] = '매우강함' # 80.0mm+

    # 🌡️ 기상청 기준 온도범주 (현재 코드 교체)  
    df_processed['temp_category'] = '적정'
    df_processed.loc[df_processed['ta'] < 0, 'temp_category'] = '추위'
    df_processed.loc[df_processed['ta'] < -10, 'temp_category'] = '한파'
    df_processed.loc[df_processed['ta'] < -15, 'temp_category'] = '혹한'
    df_processed.loc[df_processed['ta'] >= 10, 'temp_category'] = '서늘'
    df_processed.loc[df_processed['ta'] >= 20, 'temp_category'] = '적정'
    df_processed.loc[df_processed['ta'] >= 25, 'temp_category'] = '더움'
    df_processed.loc[df_processed['ta'] >= 30, 'temp_category'] = '폭염'

    # ❄️ 한파특보는 기상청 기준과 동일 (수정 불필요)
    df_processed['cold_warning_level'] = '정상'
    df_processed.loc[df_processed['ta'] <= -12, 'cold_warning_level'] = '한파주의보'
    df_processed.loc[df_processed['ta'] <= -15, 'cold_warning_level'] = '한파경보'

    # 💨 기상청 기준 풍속범주 (현재 코드 교체)
    df_processed['wind_category'] = '약함'
    df_processed.loc[df_processed['ws'] >= 3.4, 'wind_category'] = '보통'      # 3.4~7.9m/s
    df_processed.loc[df_processed['ws'] >= 8.0, 'wind_category'] = '강함'      # 8.0~13.8m/s  
    df_processed.loc[df_processed['ws'] >= 13.9, 'wind_category'] = '매우강함' # 13.9m/s+

    # 💧 기상청/보건 기준 습도범주 (현재 코드 교체)
    df_processed['humidity_category'] = '적정'
    df_processed.loc[df_processed['hm'] < 40, 'humidity_category'] = '건조'
    df_processed.loc[df_processed['hm'] >= 60, 'humidity_category'] = '습함'
    df_processed.loc[df_processed['hm'] >= 80, 'humidity_category'] = '매우습함'
    
    # 🇰🇷 한국 공휴일
    kr_holidays = holidays.KR()
    df_processed['is_holiday'] = df_processed['tm'].dt.date.apply(lambda x: x in kr_holidays)
    df_processed['holiday_type'] = df_processed['is_holiday'].map({False: '평일', True: '공휴일'})
    
    # 📅 주말 여부
    df_processed['is_weekend'] = df_processed['weekday'].isin([5, 6])
    df_processed['day_type'] = '평일'
    df_processed.loc[df_processed['is_weekend'], 'day_type'] = '주말'
    df_processed.loc[df_processed['is_holiday'], 'day_type'] = '공휴일'
    
    # 🔥 피크타임 1 (범주형) - 기존 XGBoost 코드에서 가져옴
    df_processed['peak_time1'] = '일반'
    df_processed.loc[(df_processed['hour'] >= 0) & (df_processed['hour'] <= 6), 'peak_time1'] = '새벽피크'
    df_processed.loc[(df_processed['hour'] > 6) & (df_processed['hour'] <= 11), 'peak_time1'] = '오전피크'
    df_processed.loc[(df_processed['hour'] > 11) & (df_processed['hour'] <= 18), 'peak_time1'] = '오후피크'
    df_processed.loc[(df_processed['hour'] > 18) & (df_processed['hour'] <= 23), 'peak_time1'] = '저녁피크'
    
    # 🔥 피크타임 2 (범주형) - 기존 XGBoost 코드에서 가져옴  
    df_processed['peak_time2'] = '비피크'
    df_processed.loc[(df_processed['hour'] >= 2) & (df_processed['hour'] <= 10), 'peak_time2'] = '메인피크'
    
    print("   ✅ 범주형 변수 생성 완료")
    
    return df_processed

# 범주형 변수 생성
train_df = create_categorical_features(train_df)
test_df = create_categorical_features(test_df)

print(f"\n📊 처리 후 데이터 크기:")
print(f"   훈련: {train_df.shape}")
print(f"   테스트: {test_df.shape}")

🔧 범주형 변수 생성 중...
   ❌ wd(풍향) 변수 제거
   ✅ 범주형 변수 생성 완료
🔧 범주형 변수 생성 중...
   ❌ wd(풍향) 변수 제거
   ✅ 범주형 변수 생성 완료

📊 처리 후 데이터 크기:
   훈련: (332861, 33)
   테스트: (166440, 33)


### 📈 범주형 변수 정의 및 설명

In [21]:
# 📋 범주형 변수 정의 및 설명
CATEGORICAL_FEATURES = [
    'branch_id',           # 🏢 지점/브랜치 ID (19개 지점)
    'month_cat',          # 📅 월 (1~12월)
    'weekday_name',       # 📅 요일 (월~일)
    'time_period',        # 🕐 시간대 (새벽/오전/오후/저녁)
    'season',             # ❄️ 계절 (봄/여름/가을/겨울)
    'rain_status',        # 🌧️ 시간강수 상태 (없음/약함/보통/강함/매우강함)
    'daily_rain_category', # 🌧️ 일강수량 범주 (없음/약함/보통/강함/매우강함)
    'temp_category',      # 🌡️ 온도 범주 (혹한/한파/적정/더움/폭염)
    'cold_warning_level', # ❄️ 한파특보 (정상/한파주의보/한파경보)
    'wind_category',      # 💨 풍속 범주 (약함/보통/강함/매우강함)
    'humidity_category',  # 💧 습도 범주 (건조/보통/습함/매우습함)
    'holiday_type',       # 🇰🇷 공휴일 구분 (평일/공휴일)
    'day_type',          # 📅 요일 구분 (평일/주말/공휴일)
    'peak_time1',        # 🔥 피크타임1 (새벽피크/오전피크/오후피크/저녁피크/일반)
    'peak_time2'         # 🔥 피크타임2 (메인피크/비피크)
]

print("📋 CatBoost 범주형 변수 정의 및 설명")
print("=" * 60)
print("⚠️ heating_season은 모델 분할에만 사용되며 범주형 변수에서 제외됨")

for i, cat_feature in enumerate(CATEGORICAL_FEATURES, 1):
    if cat_feature in train_df.columns:
        unique_count = train_df[cat_feature].nunique()
        unique_values = sorted(train_df[cat_feature].unique())
        
        # 값이 너무 많으면 일부만 표시
        if len(unique_values) > 10:
            display_values = f"{unique_values[:5]} ... (총 {len(unique_values)}개)"
        else:
            display_values = unique_values
        
        print(f"{i:2d}. {cat_feature:20s}: {unique_count:2d}개 범주 - {display_values}")
    else:
        print(f"{i:2d}. {cat_feature:20s}: ❌ 데이터에 없음")

# 실제 존재하는 범주형 변수만 필터링
CATEGORICAL_FEATURES = [cat for cat in CATEGORICAL_FEATURES if cat in train_df.columns]

print(f"\n✅ 실제 사용할 범주형 변수: {len(CATEGORICAL_FEATURES)}개")
print(f"📊 범주형 변수 총 유니크 조합 수: {np.prod([train_df[cat].nunique() for cat in CATEGORICAL_FEATURES[:5]]):,}개 (상위 5개만)")

📋 CatBoost 범주형 변수 정의 및 설명
⚠️ heating_season은 모델 분할에만 사용되며 범주형 변수에서 제외됨
 1. branch_id           : 19개 범주 - ['A', 'B', 'C', 'D', 'E'] ... (총 19개)
 2. month_cat           : 12개 범주 - ['1', '10', '11', '12', '2'] ... (총 12개)
 3. weekday_name        :  7개 범주 - ['금', '목', '수', '월', '일', '토', '화']
 4. time_period         :  4개 범주 - ['새벽', '오전', '오후', '저녁']
 5. season              :  4개 범주 - ['가을', '겨울', '봄', '여름']
 6. rain_status         :  5개 범주 - ['강함', '매우강함', '보통', '약함', '없음']
 7. daily_rain_category :  5개 범주 - ['강함', '매우강함', '보통', '약함', '없음']
 8. temp_category       :  7개 범주 - ['더움', '서늘', '적정', '추위', '폭염', '한파', '혹한']
 9. cold_warning_level  :  3개 범주 - ['정상', '한파경보', '한파주의보']
10. wind_category       :  3개 범주 - ['강함', '보통', '약함']
11. humidity_category   :  4개 범주 - ['건조', '매우습함', '습함', '적정']
12. holiday_type        :  2개 범주 - ['공휴일', '평일']
13. day_type            :  3개 범주 - ['공휴일', '주말', '평일']
14. peak_time1          :  4개 범주 - ['새벽피크', '오전피크', '오후피크', '저녁피크']
15. peak_time2          :  2개

## 3️⃣ 시계열 분해를 통한 고급 특성 생성

### STL 분해 (heat_demand 제외)

In [22]:
def create_time_series_features(df, target_cols=['ta', 'ws'], freq_hours=23):  # 24→23, hm→ws
    """시계열 분해를 통한 특성 생성 (STL 분해만 사용)"""
    print(f"🔄 STL 시계열 분해 특성 생성 중... (대상: {target_cols})")
    print("=" * 60)
    print("⚠️ heat_demand는 테스트 데이터에 없으므로 제외됩니다.")
    
    df_features = df.copy()
    
    # 브랜치별로 시계열 분해 수행
    for branch in tqdm(sorted(df['branch_id'].unique()), desc="브랜치별 STL 분해"):
        branch_mask = df_features['branch_id'] == branch
        branch_data = df_features[branch_mask].copy().sort_values('tm')
        
        if len(branch_data) < freq_hours * 7:  # 최소 7일 데이터 필요
            print(f"   ⚠️ 브랜치 {branch}: 데이터 부족 ({len(branch_data)}개) - 건너뜀")
            continue
        
        # 각 대상 변수별로 STL 분해 수행
        for col in target_cols:
            if col not in branch_data.columns:
                continue
                
            try:
                # 결측치 처리
                col_data = branch_data[col].interpolate().fillna(method='bfill').fillna(method='ffill')
                
                # STL 분해 (24시간 주기)
                try:
                    # 시간 인덱스 설정
                    ts_data = col_data.copy()
                    ts_data.index = pd.to_datetime(branch_data['tm'])
                    
                    # STL 분해
                    stl = STL(ts_data, seasonal=freq_hours, robust=True)
                    stl_result = stl.fit()
                    
                    # STL 결과 저장
                    indices = branch_data.index
                    df_features.loc[indices, f'{col}_stl_trend'] = stl_result.trend.values
                    df_features.loc[indices, f'{col}_stl_seasonal'] = stl_result.seasonal.values
                    df_features.loc[indices, f'{col}_stl_resid'] = stl_result.resid.values
                    
                    # 추가 파생 변수
                    df_features.loc[indices, f'{col}_detrend'] = col_data.values - stl_result.trend.values
                    df_features.loc[indices, f'{col}_seasonal_strength'] = np.abs(stl_result.seasonal.values)
                    
                    # 계절성 변동 지표
                    seasonal_std = np.std(stl_result.seasonal.values)
                    df_features.loc[indices, f'{col}_seasonal_volatility'] = seasonal_std
                    
                except Exception as e:
                    print(f"      ⚠️ STL 분해 실패 ({col}): {str(e)[:50]}")
                    continue
                
            except Exception as e:
                print(f"   ❌ 브랜치 {branch} {col} 분해 실패: {str(e)[:50]}")
                continue
    
    # 생성된 시계열 특성 목록
    time_series_features = [col for col in df_features.columns 
                           if any(pattern in col for pattern in ['_stl_', '_detrend', '_seasonal_strength', '_seasonal_volatility'])]
    
    print(f"\n✅ STL 시계열 분해 완료!")
    print(f"📊 생성된 시계열 특성: {len(time_series_features)}개")
    print(f"📋 특성 목록: {time_series_features[:10]}{'...' if len(time_series_features) > 10 else ''}")
    
    return df_features, time_series_features

# 시계열 분해 특성 생성 (heat_demand 제외)
print("🚀 STL 시계열 분해 시작...")
train_df, train_ts_features = create_time_series_features(train_df, target_cols=['ta', 'ws'])
test_df, test_ts_features = create_time_series_features(test_df, target_cols=['ta', 'ws'])

print(f"\n📊 시계열 분해 후 데이터 크기:")
print(f"   훈련: {train_df.shape}")
print(f"   테스트: {test_df.shape}")

🚀 STL 시계열 분해 시작...
🔄 STL 시계열 분해 특성 생성 중... (대상: ['ta', 'ws'])
⚠️ heat_demand는 테스트 데이터에 없으므로 제외됩니다.


브랜치별 STL 분해:   0%|          | 0/19 [00:00<?, ?it/s]


✅ STL 시계열 분해 완료!
📊 생성된 시계열 특성: 12개
📋 특성 목록: ['ta_stl_trend', 'ta_stl_seasonal', 'ta_stl_resid', 'ta_detrend', 'ta_seasonal_strength', 'ta_seasonal_volatility', 'ws_stl_trend', 'ws_stl_seasonal', 'ws_stl_resid', 'ws_detrend']...
🔄 STL 시계열 분해 특성 생성 중... (대상: ['ta', 'ws'])
⚠️ heat_demand는 테스트 데이터에 없으므로 제외됩니다.


브랜치별 STL 분해:   0%|          | 0/19 [00:00<?, ?it/s]


✅ STL 시계열 분해 완료!
📊 생성된 시계열 특성: 12개
📋 특성 목록: ['ta_stl_trend', 'ta_stl_seasonal', 'ta_stl_resid', 'ta_detrend', 'ta_seasonal_strength', 'ta_seasonal_volatility', 'ws_stl_trend', 'ws_stl_seasonal', 'ws_stl_resid', 'ws_detrend']...

📊 시계열 분해 후 데이터 크기:
   훈련: (332861, 45)
   테스트: (166440, 45)


## 4️⃣ 추가 특성 엔지니어링

In [23]:
def create_additional_features(df):
    """추가 수치형 특성 생성"""
    print("🔧 추가 특성 엔지니어링...")
    print("⚠️ heat_demand 관련 특성은 테스트 데이터에 없으므로 제외됩니다.")
    
    df_enhanced = df.copy()
    
    # HDD/CDD (Heating/Cooling Degree Days)
    df_enhanced['HDD18'] = np.maximum(0, 18 - df_enhanced['ta'])
    df_enhanced['CDD18'] = np.maximum(0, df_enhanced['ta'] - 18)
    df_enhanced['HDD20'] = np.maximum(0, 20 - df_enhanced['ta'])
    df_enhanced['CDD20'] = np.maximum(0, df_enhanced['ta'] - 20)
    
    # 체감온도 계산
    def calculate_apparent_temp(ta, hm, ws):
        """체감온도 계산 (계절별)"""
        # 여름철 불쾌지수 기반
        summer_at = 0.81 * ta + 0.01 * hm * (0.99 * ta - 14.3) + 46.3
        # 겨울철 wind chill 기반
        winter_at = 13.12 + 0.6215 * ta - 11.37 * (ws * 3.6)**0.16 + 0.3965 * ta * (ws * 3.6)**0.16
        
        return np.where(ta > 10, summer_at, winter_at)
    
    df_enhanced['apparent_temp'] = calculate_apparent_temp(
        df_enhanced['ta'], df_enhanced['hm'], df_enhanced['ws']
    )
    
    # 브랜치별 온도 지연 특성 (1, 3, 6, 24시간)
    for lag in [1, 3, 6, 24]:
        df_enhanced[f'ta_lag_{lag}h'] = df_enhanced.groupby('branch_id')['ta'].shift(lag)
        df_enhanced[f'hm_lag_{lag}h'] = df_enhanced.groupby('branch_id')['hm'].shift(lag)
    
    # 브랜치별 온도/풍속 이동평균 (6시간, 12시간, 24시간)
    for window in [6, 12, 24]:
        df_enhanced[f'ta_ma_{window}h'] = df_enhanced.groupby('branch_id')['ta'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        df_enhanced[f'ws_ma_{window}h'] = df_enhanced.groupby('branch_id')['ws'].transform(  # ← ws
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # 온도 차분 (시간별 변화율)
    df_enhanced['ta_diff_1h'] = df_enhanced.groupby('branch_id')['ta'].diff(1)
    df_enhanced['ta_diff_3h'] = df_enhanced.groupby('branch_id')['ta'].diff(3)
    df_enhanced['ta_diff_6h'] = df_enhanced.groupby('branch_id')['ta'].diff(6)
    
    # 풍속 차분
    df_enhanced['ws_diff_1h'] = df_enhanced.groupby('branch_id')['ws'].diff(1)  # ← ws
    df_enhanced['ws_diff_3h'] = df_enhanced.groupby('branch_id')['ws'].diff(3)  # ← ws
    
    
    # 일교차 및 일풍속차 (당일 최고-최저)
    daily_stats = df_enhanced.groupby(['branch_id', df_enhanced['tm'].dt.date]).agg({
        'ta': ['min', 'max', 'mean', 'std'],
        'ws': ['min', 'max', 'mean', 'std']  # ← ws
    }).round(2)
    daily_stats.columns = ['daily_ta_min', 'daily_ta_max', 'daily_ta_mean', 'daily_ta_std',
                          'daily_ws_min', 'daily_ws_max', 'daily_ws_mean', 'daily_ws_std']  # ← ws
    daily_stats['daily_temp_range'] = daily_stats['daily_ta_max'] - daily_stats['daily_ta_min']
    daily_stats['daily_ws_range'] = daily_stats['daily_ws_max'] - daily_stats['daily_ws_min']  # ← ws
    
    # 일별 통계를 원본 데이터에 병합
    df_enhanced = df_enhanced.merge(
        daily_stats.reset_index(), 
        left_on=['branch_id', df_enhanced['tm'].dt.date], 
        right_on=['branch_id', 'tm'], 
        how='left',
        suffixes=('', '_daily')
    )
    
    # 순환 인코딩 (시간의 주기성 반영)
    df_enhanced['hour_sin'] = np.sin(2 * np.pi * df_enhanced['hour'] / 24)
    df_enhanced['hour_cos'] = np.cos(2 * np.pi * df_enhanced['hour'] / 24)
    df_enhanced['day_sin'] = np.sin(2 * np.pi * df_enhanced['dayofyear'] / 365)
    df_enhanced['day_cos'] = np.cos(2 * np.pi * df_enhanced['dayofyear'] / 365)
    df_enhanced['month_sin'] = np.sin(2 * np.pi * df_enhanced['month'] / 12)
    df_enhanced['month_cos'] = np.cos(2 * np.pi * df_enhanced['month'] / 12)
    
    # 온도-풍속 상호작용 특성 (ta_hm_interaction 제거)
    df_enhanced['ta_ws_interaction'] = df_enhanced['ta'] * df_enhanced['ws']
    
    # 결측치 처리 (생성된 특성들)
    numeric_cols = df_enhanced.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_enhanced[col].isna().sum() > 0:
            df_enhanced[col] = df_enhanced.groupby('branch_id')[col].transform(
                lambda x: x.fillna(x.mean())
            ).fillna(df_enhanced[col].mean())
    
    print(f"   ✅ 추가 특성 생성 완료: {df_enhanced.shape[1] - df.shape[1]}개 추가")
    
    return df_enhanced

# 추가 특성 생성
train_df = create_additional_features(train_df)
test_df = create_additional_features(test_df)

print(f"\n📊 최종 특성 생성 후 데이터 크기:")
print(f"   훈련: {train_df.shape}")
print(f"   테스트: {test_df.shape}")

🔧 추가 특성 엔지니어링...
⚠️ heat_demand 관련 특성은 테스트 데이터에 없으므로 제외됩니다.
   ✅ 추가 특성 생성 완료: 42개 추가
🔧 추가 특성 엔지니어링...
⚠️ heat_demand 관련 특성은 테스트 데이터에 없으므로 제외됩니다.
   ✅ 추가 특성 생성 완료: 42개 추가

📊 최종 특성 생성 후 데이터 크기:
   훈련: (332861, 87)
   테스트: (166440, 87)


## 5️⃣ 데이터 분할 (난방/비난방 시즌)

In [24]:
# 시즌별 데이터 분할
def split_by_season(df):
    """난방/비난방 시즌별로 데이터 분할"""
    heating_data = df[df['heating_season'] == 1].copy()
    non_heating_data = df[df['heating_season'] == 0].copy()
    
    return heating_data, non_heating_data

# 훈련/테스트 데이터 시즌별 분할
train_heating, train_non_heating = split_by_season(train_df)
test_heating, test_non_heating = split_by_season(test_df)

print("📊 시즌별 데이터 분할 결과:")
print("=" * 50)
print(f"🔥 난방시즌:")
print(f"   훈련: {len(train_heating):,}개 ({train_heating['tm'].min().strftime('%Y-%m-%d')} ~ {train_heating['tm'].max().strftime('%Y-%m-%d')})")
print(f"   테스트: {len(test_heating):,}개 ({test_heating['tm'].min().strftime('%Y-%m-%d')} ~ {test_heating['tm'].max().strftime('%Y-%m-%d')})")

print(f"\n❄️ 비난방시즌:")
print(f"   훈련: {len(train_non_heating):,}개 ({train_non_heating['tm'].min().strftime('%Y-%m-%d')} ~ {train_non_heating['tm'].max().strftime('%Y-%m-%d')})")
print(f"   테스트: {len(test_non_heating):,}개 ({test_non_heating['tm'].min().strftime('%Y-%m-%d')} ~ {test_non_heating['tm'].max().strftime('%Y-%m-%d')})")

# 시즌별 기본 통계
print(f"\n📈 시즌별 열수요 통계:")
print(f"   난방시즌 평균: {train_heating['heat_demand'].mean():.2f}")
print(f"   비난방시즌 평균: {train_non_heating['heat_demand'].mean():.2f}")
print(f"   비율: {train_heating['heat_demand'].mean() / train_non_heating['heat_demand'].mean():.1f}배")

📊 시즌별 데이터 분할 결과:
🔥 난방시즌:
   훈련: 193,325개 (2021-01-01 ~ 2022-12-31)
   테스트: 96,672개 (2023-01-01 ~ 2023-12-31)

❄️ 비난방시즌:
   훈련: 139,536개 (2021-05-01 ~ 2022-09-30)
   테스트: 69,768개 (2023-05-01 ~ 2023-09-30)

📈 시즌별 열수요 통계:
   난방시즌 평균: 138.89
   비난방시즌 평균: 40.06
   비율: 3.5배


## 6️⃣ CatBoost 모델 클래스 정의

In [25]:
class OptimalCatBoostModel:
    def __init__(self, model_name, categorical_features=None):
        self.model_name = model_name
        self.model = None
        self.best_params = None
        self.feature_cols = None
        self.categorical_features = categorical_features or []
        self.study = None
        self.best_score = None
        
    def define_feature_columns(self, df):
        """특성 컬럼 정의"""
        exclude_cols = ['tm', 'heat_demand', 'level_0', 'level_1', 'tm_daily']
        self.feature_cols = [col for col in df.columns 
                           if col not in exclude_cols and not col.startswith('Unnamed')]
        
        # 범주형 특성 인덱스 계산
        self.categorical_indices = []
        for cat_feature in self.categorical_features:
            if cat_feature in self.feature_cols:
                self.categorical_indices.append(self.feature_cols.index(cat_feature))
        
        print(f"   📋 {self.model_name}: {len(self.feature_cols)}개 특성 사용")
        print(f"   📊 범주형 특성: {len(self.categorical_indices)}개")
        
        return self.feature_cols

    def get_temporal_split_indices_advanced(self, df, test_size=0.2, min_val_samples=10):
        """월별 + 시간대별 층화추출 CV"""
        df_copy = df.copy()
        
        # 월별, 시간별 층화추출
        month_counts = df_copy['month'].value_counts()
        valid_months = month_counts[month_counts >= min_val_samples * 2].index
        
        if len(valid_months) < 3:
            # 단순 시간순 분할
            split_idx = int(len(df) * (1 - test_size))
            return df.index[:split_idx], df.index[split_idx:]
        
        train_indices = []
        val_indices = []
        
        for month in valid_months:
            month_data = df_copy[df_copy['month'] == month]
            val_size = max(min_val_samples, int(len(month_data) * test_size))
            
            # 시간대별 비례 추출
            hour_proportions = month_data['hour'].value_counts(normalize=True).sort_index()
            
            month_val_indices = []
            month_train_indices = []
            
            for hour, proportion in hour_proportions.items():
                hour_data = month_data[month_data['hour'] == hour]
                if len(hour_data) == 0:
                    continue
                    
                hour_val_size = max(1, int(val_size * proportion))
                hour_val_size = min(hour_val_size, len(hour_data) - 1)
                
                if hour_val_size > 0 and len(hour_data) > 1:
                    hour_indices = hour_data.index.tolist()
                    np.random.seed(42 + hour + month)  # 재현 가능한 랜덤
                    
                    if len(hour_indices) > hour_val_size:
                        hour_val_sample = np.random.choice(hour_indices, size=hour_val_size, replace=False)
                        hour_train_sample = [idx for idx in hour_indices if idx not in hour_val_sample]
                    else:
                        hour_val_sample = hour_indices[:1]
                        hour_train_sample = hour_indices[1:]
                    
                    month_val_indices.extend(hour_val_sample)
                    month_train_indices.extend(hour_train_sample)
                else:
                    month_train_indices.extend(hour_data.index.tolist())
            
            train_indices.extend(month_train_indices)
            val_indices.extend(month_val_indices)
        
        print(f"      🎯 월별+시간대별 층화추출: {len(valid_months)}개월, 검증 {len(val_indices)}개")
        return train_indices, val_indices

    def objective(self, trial, X_train, y_train, X_val, y_val):
        """Optuna 목적 함수 - CatBoost 최적화"""
        
        # CatBoost 하이퍼파라미터 최적화
        params = {
        'iterations': trial.suggest_int('iterations', 300, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 12),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'od_wait': trial.suggest_int('od_wait', 20, 100),
        # subsample 제거됨!
        
        # 고정 파라미터
        'objective': 'RMSE',
        'eval_metric': 'RMSE',
        'od_type': 'Iter',
        'boosting_type': 'Ordered',
        'bootstrap_type': 'Bayesian',
        'leaf_estimation_method': 'Newton',
        'grow_policy': 'SymmetricTree',
        'penalties_coefficient': 1,
        'feature_border_type': 'GreedyLogSum',
        'random_seed': 42,
        'thread_count': -1,
        'verbose': False
        }
        
        # CatBoost 모델 생성
        model = CatBoostRegressor(**params)
        
        # 훈련 (조기 중단 포함)
        model.fit(
            X_train, y_train,
            eval_set=(X_val, y_val),
            cat_features=self.categorical_indices,
            early_stopping_rounds=50,
            verbose=False,
            plot=False
        )
        
        # 검증 예측 및 RMSE 계산
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        return rmse
    
    def fit(self, df, target_col='heat_demand', n_trials=100):
        """CatBoost 모델 최적화 훈련"""
        print(f"\n🔥 {self.model_name} CatBoost 최적화 시작...")
        
        # 1. 고급 CV 분할
        train_indices, val_indices = self.get_temporal_split_indices_advanced(df, test_size=0.2)
        
        # 2. 특성 컬럼 정의
        self.define_feature_columns(df)
        
        # 3. 데이터 준비
        X = df[self.feature_cols].copy()
        y = df[target_col].copy()
        
        # 범주형 변수 문자열로 변환
        for cat_feature in self.categorical_features:
            if cat_feature in X.columns:
                X[cat_feature] = X[cat_feature].astype(str)
        
        # 결측치 처리
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        # 4. Train/Validation 분할
        X_train = X.loc[train_indices]
        y_train = y.loc[train_indices]
        X_val = X.loc[val_indices]
        y_val = y.loc[val_indices]
        
        print(f"      📊 훈련: {len(X_train):,}개, 검증: {len(X_val):,}개")
        print(f"      📋 범주형 특성: {len(self.categorical_indices)}개")
        
        # 5. Optuna 최적화
        print(f"   🎯 Optuna 하이퍼파라미터 최적화 ({n_trials}회 시도)...")
        
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(
                seed=42,
                n_startup_trials=20,
                n_ei_candidates=24,
                multivariate=True,
                constant_liar=True
            ),
            study_name=f"catboost_{self.model_name}"
        )
        
        # 최적화 실행
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=True
        )
        
        # 6. 최고 모델 훈련
        self.best_score = study.best_value
        self.best_params = study.best_params.copy()
        
        # 최적 파라미터로 최종 모델 훈련
        final_params = self.best_params.copy()
        final_params.update({
            'objective': 'RMSE',
            'eval_metric': 'RMSE',
            'od_type': 'Iter',
            'boosting_type': 'Ordered',
            'bootstrap_type': 'Bayesian',
            'random_seed': 42,
            'thread_count': -1,
            'verbose': False
        })
        
        self.model = CatBoostRegressor(**final_params)
        self.model.fit(
            X, y,
            cat_features=self.categorical_indices,
            verbose=False
        )
        
        # 성능 검증
        val_pred = self.model.predict(X_val)
        final_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        
        print(f"   📈 최적화 완료!")
        print(f"   🏆 Best RMSE: {self.best_score:.4f}")
        print(f"   📊 Final RMSE: {final_rmse:.4f}")
        print(f"   ⚡ 최적 파라미터: lr={final_params['learning_rate']:.4f}, depth={final_params['depth']}, iter={final_params['iterations']}")
        
        self.study = study
    
    def predict(self, df):
        """예측"""
        X = df[self.feature_cols].copy()
        
        # 범주형 변수 문자열로 변환
        for cat_feature in self.categorical_features:
            if cat_feature in X.columns:
                X[cat_feature] = X[cat_feature].astype(str)
        
        X = X.fillna(0)
        predictions = self.model.predict(X)
        return np.maximum(predictions, 0)  # 음수값 제거
    
    def get_feature_importance(self, top_n=20):
        """특성 중요도 반환"""
        if self.model is None:
            return None
            
        importance = self.model.get_feature_importance()
        feature_importance = pd.DataFrame({
            'feature': self.feature_cols,
            'importance': importance
        }).sort_values('importance', ascending=False)
        
        return feature_importance.head(top_n)

print("🚀 CatBoost 모델 클래스 정의 완료!")

🚀 CatBoost 모델 클래스 정의 완료!


## 7️⃣ 시즌별 모델 훈련 (2개 모델)

In [ ]:
# 시즌별 CatBoost 모델 훈련
print("🚀 시즌별 CatBoost 모델 훈련 시작!")
print("🎯 목표: 난방/비난방 시즌별 최적 성능 달성")
print("=" * 60)

models = {}
training_results = {}
n_trials_per_model = 3  # 충분한 최적화 시도 # 원래는 100이었으나 trail 한번에 10분 소요되서 일단 축약해서 확인함.

start_time = datetime.now()

# 시즌별 데이터와 모델명 정의
season_data = {
    '난방시즌': train_heating,
    '비난방시즌': train_non_heating
}

# 각 시즌별 모델 훈련
for i, (season_name, train_data) in enumerate(season_data.items(), 1):
    print(f"\n[{i}/2] 🔥 {season_name} 모델 훈련")
    print(f"      📊 데이터: {len(train_data):,}개")
    print(f"      📈 평균 열수요: {train_data['heat_demand'].mean():.2f}")
    
    try:
        # CatBoost 모델 생성
        model = OptimalCatBoostModel(
            model_name=season_name,
            categorical_features=CATEGORICAL_FEATURES
        )
        
        model_start = datetime.now()
        model.fit(train_data, n_trials=n_trials_per_model)
        model_time = (datetime.now() - model_start).total_seconds()
        
        # 성공
        models[season_name] = model
        
        training_results[season_name] = {
            'data_size': len(train_data),
            'training_time': model_time,
            'best_score': model.best_score,
            'best_params': model.best_params,
            'feature_count': len(model.feature_cols),
            'categorical_count': len(model.categorical_indices),
            'optimization_success': True,
            'n_trials': n_trials_per_model
        }
        
        print(f"      ✅ 성공 | RMSE: {model.best_score:.4f} | ⏱️ {model_time:.1f}초")
        print(f"      📋 특성: {len(model.feature_cols)}개 (범주형 {len(model.categorical_indices)}개)")
        
        # 특성 중요도 출력
        importance = model.get_feature_importance(top_n=10)
        if importance is not None:
            print(f"      🏆 Top 5 특성: {', '.join(importance.head(5)['feature'].tolist())}")
        
    except Exception as e:
        print(f"      ❌ 실패: {str(e)}")
        
        training_results[season_name] = {
            'data_size': len(train_data),
            'training_time': 0,
            'best_score': None,
            'best_params': None,
            'optimization_success': False,
            'error': str(e)
        }

total_time = (datetime.now() - start_time).total_seconds()

print(f"\n" + "=" * 60)
print(f"🎉 CatBoost 모델 훈련 완료!")
print(f"⏱️  총 소요 시간: {total_time/60:.1f}분")
print(f"📊 훈련 결과:")

success_count = len([r for r in training_results.values() if r.get('optimization_success', False)])
print(f"   ✅ 성공: {success_count}/2개 모델")

if success_count > 0:
    successful_models = {k: v for k, v in training_results.items() 
                        if v.get('optimization_success', False)}
    
    scores = [v['best_score'] for v in successful_models.values()]
    best_model = min(successful_models.items(), key=lambda x: x[1]['best_score'])
    
    print(f"\n🏆 성능 통계:")
    print(f"   최고 RMSE: {min(scores):.4f}")
    print(f"   평균 RMSE: {np.mean(scores):.4f}")
    print(f"   최고 모델: {best_model[0]} (RMSE: {best_model[1]['best_score']:.4f})")
    
    # 시즌별 성능 비교
    for season_name, result in successful_models.items():
        print(f"   {season_name}: RMSE {result['best_score']:.4f} | 데이터 {result['data_size']:,}개")

print("=" * 60)

🚀 시즌별 CatBoost 모델 훈련 시작!
🎯 목표: 난방/비난방 시즌별 최적 성능 달성

[1/2] 🔥 난방시즌 모델 훈련
      📊 데이터: 193,325개
      📈 평균 열수요: 138.89

🔥 난방시즌 CatBoost 최적화 시작...
      🎯 월별+시간대별 층화추출: 7개월, 검증 38588개
   📋 난방시즌: 84개 특성 사용
   📊 범주형 특성: 15개


[I 2025-06-19 11:47:25,623] A new study created in memory with name: catboost_난방시즌


      📊 훈련: 154,737개, 검증: 38,588개
      📋 범주형 특성: 15개
   🎯 Optuna 하이퍼파라미터 최적화 (3회 시도)...


  0%|          | 0/3 [00:00<?, ?it/s]

[I 2025-06-19 11:58:55,380] Trial 0 finished with value: 13.27449837210107 and parameters: {'iterations': 937, 'learning_rate': 0.2536999076681772, 'depth': 10, 'l2_leaf_reg': 0.24810409748678125, 'border_count': 66, 'bagging_temperature': 0.15599452033620265, 'random_strength': 0.0017073967431528124, 'od_wait': 90}. Best is trial 0 with value: 13.27449837210107.
[I 2025-06-19 12:01:50,720] Trial 1 finished with value: 16.77621976170882 and parameters: {'iterations': 1322, 'learning_rate': 0.11114989443094977, 'depth': 4, 'l2_leaf_reg': 7.579479953348009, 'border_count': 218, 'bagging_temperature': 0.21233911067827616, 'random_strength': 0.005337032762603957, 'od_wait': 34}. Best is trial 0 with value: 13.27449837210107.


## 8️⃣ 특성 중요도 분석

In [ ]:
# 특성 중요도 분석 및 시각화
print("📊 CatBoost 특성 중요도 분석")
print("=" * 60)

# 각 모델별 특성 중요도 분석
importance_results = {}

for season_name, model in models.items():
    print(f"\n🔥 {season_name} 모델 특성 중요도:")
    
    importance = model.get_feature_importance(top_n=20)
    if importance is not None:
        importance_results[season_name] = importance
        
        print(f"   📋 Top 20 특성:")
        for idx, row in importance.iterrows():
            # 범주형 여부 표시
            cat_mark = "🏷️" if row['feature'] in CATEGORICAL_FEATURES else "📊"
            print(f"      {idx+1:2d}. {cat_mark} {row['feature']:25s}: {row['importance']:8.2f}")
    else:
        print(f"   ❌ 특성 중요도를 가져올 수 없습니다.")

# 시각화
if len(importance_results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    for idx, (season_name, importance) in enumerate(importance_results.items()):
        # Top 15 특성만 시각화
        top_15 = importance.head(15)
        
        # 범주형/수치형 구분 색상
        colors = ['#FF6B6B' if feat in CATEGORICAL_FEATURES else '#4ECDC4' 
                 for feat in top_15['feature']]
        
        bars = axes[idx].barh(range(len(top_15)), top_15['importance'], color=colors)
        axes[idx].set_yticks(range(len(top_15)))
        axes[idx].set_yticklabels(top_15['feature'])
        axes[idx].set_xlabel('Feature Importance')
        axes[idx].set_title(f'{season_name} 모델\nTop 15 Feature Importance')
        axes[idx].invert_yaxis()
        
        # 값 표시
        for i, bar in enumerate(bars):
            width = bar.get_width()
            axes[idx].text(width + width*0.01, bar.get_y() + bar.get_height()/2, 
                          f'{width:.1f}', ha='left', va='center', fontsize=8)
    
    # 범례
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#FF6B6B', label='범주형 변수'),
                      Patch(facecolor='#4ECDC4', label='수치형 변수')]
    fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.02), ncol=2)
    
    plt.tight_layout()
    plt.savefig('catboost_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n💾 특성 중요도 시각화 저장: catboost_feature_importance.png")

# 특성 중요도 CSV 저장
for season_name, importance in importance_results.items():
    filename = f"feature_importance_{season_name}.csv"
    importance.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"💾 {filename} 저장 완료")

print(f"\n✅ 특성 중요도 분석 완료!")

## 9️⃣ 테스트 데이터 예측

In [ ]:
# 테스트 데이터 예측
print("🎯 테스트 데이터 예측 시작...")
print("=" * 40)

# 시즌별 테스트 데이터 정의
test_season_data = {
    '난방시즌': test_heating,
    '비난방시즌': test_non_heating
}

# 예측 결과 저장
predictions = {}
prediction_stats = {}

# 각 시즌별 모델로 예측
for season_name, model in models.items():
    if season_name in test_season_data:
        test_data = test_season_data[season_name]
        
        print(f"📊 {season_name}: {len(test_data):,}개 데이터 예측 중...")
        
        try:
            pred = model.predict(test_data)
            predictions[season_name] = {
                'data': test_data,
                'predictions': pred
            }
            
            # 예측 통계
            prediction_stats[season_name] = {
                'count': len(pred),
                'mean': np.mean(pred),
                'std': np.std(pred),
                'min': np.min(pred),
                'max': np.max(pred)
            }
            
            print(f"   ✅ 완료: 평균={np.mean(pred):.2f}, 범위=[{np.min(pred):.2f}, {np.max(pred):.2f}]")
            
        except Exception as e:
            print(f"   ❌ 예측 실패: {str(e)}")
            # 기본값으로 0 할당
            predictions[season_name] = {
                'data': test_data,
                'predictions': np.zeros(len(test_data))
            }
    else:
        print(f"⚠️ {season_name}: 대응하는 테스트 데이터 없음")

print(f"\n✅ 예측 완료: {len(predictions)}개 모델")

# 예측 통계 요약
if prediction_stats:
    stats_df = pd.DataFrame(prediction_stats).T
    print(f"\n📈 예측값 통계 요약:")
    print(f"   전체 예측 개수: {stats_df['count'].sum():,}개")
    print(f"   평균 예측값 범위: [{stats_df['mean'].min():.2f}, {stats_df['mean'].max():.2f}]")
    print(f"   최대 예측값: {stats_df['max'].max():.2f}")
    
    for season_name, stats in prediction_stats.items():
        print(f"   {season_name}: 평균 {stats['mean']:.2f} ± {stats['std']:.2f}")

## 🔟 예측 결과 통합 및 최종 평가

In [ ]:
# 예측 결과를 원본 test_df 순서에 맞게 통합
print("🔄 예측 결과 통합 중...")

# 결과를 저장할 배열 초기화
final_predictions = np.zeros(len(test_df))
prediction_counts = np.zeros(len(test_df))

# 각 시즌별 예측 결과를 해당 인덱스에 할당
for season_name, pred_info in predictions.items():
    test_data = pred_info['data']
    pred_values = pred_info['predictions']
    
    # 원본 test_df에서 해당 시즌 데이터의 인덱스 찾기
    season_num = 1 if season_name == '난방시즌' else 0
    mask = test_df['heating_season'] == season_num
    indices = test_df[mask].index.tolist()
    
    print(f"📊 {season_name}: {len(indices)}개 인덱스에 할당")
    
    # 예측값 할당
    if len(indices) == len(pred_values):
        for i, idx in enumerate(indices):
            final_predictions[idx] = pred_values[i]
            prediction_counts[idx] += 1
    else:
        print(f"   ⚠️ 크기 불일치: 인덱스 {len(indices)}개 vs 예측값 {len(pred_values)}개")
        min_len = min(len(indices), len(pred_values))
        for i in range(min_len):
            final_predictions[indices[i]] = pred_values[i]
            prediction_counts[indices[i]] += 1

# 예측되지 않은 데이터 확인
unassigned_count = np.sum(prediction_counts == 0)
if unassigned_count > 0:
    print(f"⚠️ 예측되지 않은 데이터: {unassigned_count}개 (0으로 유지)")

print(f"\n✅ 예측 결과 통합 완료")
print(f"   📊 총 예측 개수: {len(final_predictions):,}개")
print(f"   📈 예측값 통계: 평균={np.mean(final_predictions):.2f}, 최대={np.max(final_predictions):.2f}")

# 최종 결과를 test_df에 추가
result_df = test_df.copy()
result_df['pred_heat_demand'] = np.maximum(final_predictions, 0).round(1)

# CSV 파일 저장
output_filename = 'catboost_season_predictions.csv'
result_df[['tm', 'branch_id', 'heat_demand', 'pred_heat_demand', 'heating_season']].to_csv(
    output_filename, index=False, encoding='utf-8-sig'
)

print(f"\n📁 결과 파일 저장: {output_filename}")

In [ ]:
# RMSE 성능 평가 (실제값이 있는 경우)
if 'heat_demand' in test_df.columns:
    print(f"\n📊 CatBoost 모델 성능 평가")
    print("=" * 60)
    
    # 전체 RMSE
    y_true = test_df['heat_demand'].values
    y_pred = result_df['pred_heat_demand'].values
    
    # 유효한 데이터만 사용
    valid_mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = y_true[valid_mask]
    y_pred_clean = y_pred[valid_mask]
    
    overall_rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    overall_mae = mean_absolute_error(y_true_clean, y_pred_clean)
    correlation = np.corrcoef(y_true_clean, y_pred_clean)[0, 1]
    
    print(f"🏆 전체 성능:")
    print(f"   RMSE: {overall_rmse:.4f}")
    print(f"   MAE:  {overall_mae:.4f}")
    print(f"   상관계수: {correlation:.4f}")
    print(f"   유효 데이터: {len(y_true_clean):,}개")
    
    # 시즌별 RMSE
    print(f"\n📈 시즌별 성능:")
    season_names = {0: '비난방시즌', 1: '난방시즌'}
    
    for season in [0, 1]:
        mask = (test_df['heating_season'] == season) & valid_mask
        if np.sum(mask) > 0:
            season_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
            season_mae = mean_absolute_error(y_true[mask], y_pred[mask])
            season_corr = np.corrcoef(y_true[mask], y_pred[mask])[0, 1] if np.sum(mask) > 1 else 0
            
            print(f"   {season_names[season]:8s}: RMSE={season_rmse:7.4f} | MAE={season_mae:7.4f} | 상관={season_corr:6.3f} | {np.sum(mask):,}개")
    
    # 브랜치별 성능 (상위/하위 5개)
    print(f"\n📊 브랜치별 RMSE 성능:")
    branch_results = []
    
    for branch in sorted(test_df['branch_id'].unique()):
        mask = (test_df['branch_id'] == branch) & valid_mask
        if np.sum(mask) > 1:
            branch_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
            branch_results.append((branch, branch_rmse, np.sum(mask)))
    
    if branch_results:
        branch_results.sort(key=lambda x: x[1])  # RMSE 기준 정렬
        
        print(f"   🥇 RMSE 우수 브랜치 (Top 5):")
        for i, (branch, rmse, count) in enumerate(branch_results[:5], 1):
            print(f"      {i}. 브랜치 {branch}: RMSE={rmse:7.4f} | {count:,}개")
        
        print(f"   🥉 RMSE 개선 필요 브랜치 (Bottom 5):")
        for i, (branch, rmse, count) in enumerate(branch_results[-5:], 1):
            print(f"      {i}. 브랜치 {branch}: RMSE={rmse:7.4f} | {count:,}개")
    
    # 실제값 vs 예측값 시각화
    plt.figure(figsize=(15, 5))
    
    # 전체 산점도
    plt.subplot(1, 3, 1)
    plt.scatter(y_true_clean, y_pred_clean, alpha=0.5, s=1)
    plt.plot([y_true_clean.min(), y_true_clean.max()], [y_true_clean.min(), y_true_clean.max()], 'r--', lw=2)
    plt.xlabel('실제값')
    plt.ylabel('예측값')
    plt.title(f'전체 예측 성능\nRMSE: {overall_rmse:.4f}')
    plt.grid(True, alpha=0.3)
    
    # 시즌별 박스플롯
    plt.subplot(1, 3, 2)
    season_errors = []
    season_labels = []
    
    for season in [0, 1]:
        mask = (test_df['heating_season'] == season) & valid_mask
        if np.sum(mask) > 0:
            errors = np.abs(y_true[mask] - y_pred[mask])
            season_errors.append(errors)
            season_labels.append(season_names[season])
    
    plt.boxplot(season_errors, labels=season_labels)
    plt.ylabel('절대오차')
    plt.title('시즌별 예측 오차 분포')
    plt.grid(True, alpha=0.3)
    
    # 시계열 예측 결과 (일부 샘플)
    plt.subplot(1, 3, 3)
    sample_branch = test_df['branch_id'].iloc[0]
    sample_mask = (test_df['branch_id'] == sample_branch) & valid_mask
    sample_data = test_df[sample_mask].sort_values('tm').head(168)  # 1주일
    
    if len(sample_data) > 0:
        plt.plot(sample_data['tm'], sample_data['heat_demand'], label='실제값', linewidth=2)
        plt.plot(sample_data['tm'], result_df.loc[sample_data.index, 'pred_heat_demand'], 
                label='예측값', linewidth=2, alpha=0.8)
        plt.xlabel('시간')
        plt.ylabel('열수요')
